# Case Study 3 — Unsupervised: Segmenting Traffic Analysis Zones for Transport Planning

**Author:** Piyush Jaangid  |  **Domain:** Urban Transport Planning / 4-Stage Modelling Context  |  **Type:** Unsupervised Clustering

---

## Problem Statement (Step 0)

A city's Comprehensive Mobility Plan (CMP) is built on a **4-stage transport model**: Trip Generation → Trip Distribution → Mode Choice → Trip Assignment [1][2]. Stage 1 — Trip Generation — produces trips per zone using regression equations like:

```
Productions_i = α_1·Population_i + α_2·Workers_i + ...
Attractions_i = β_1·Employment_i + β_2·Retail_Employment_i + ...
```

**Problem:** these regression coefficients are *zone-type specific*. A "CBD" zone behaves nothing like a "residential outskirt" zone — using one set of coefficients for both produces garbage forecasts. CMPs traditionally classify zones manually using planner judgement [2].

> **Use unsupervised clustering to discover zone segments objectively from the data, so trip-generation coefficients can be calibrated per cluster.**

| Item | Specification |
|---|---|
| **Goal** | Group similar zones, no labels available → **Unsupervised clustering** |
| **Input X** | Population, households, employment, retail employment, students, median income |
| **Output** | Cluster label per zone |
| **Constraint** | Clusters must be interpretable for planners — this isn't a black-box exercise |
| **Business KPI** | Silhouette coefficient (cluster cohesion); planner sign-off (qualitative) |

**Why this matters commercially:** WRI India and ITDP use exactly this kind of segmentation in CMP studies for cities like Pune, Kochi, and Bhubaneswar [3][4]. A poorly segmented model overestimates demand by 20–40% in greenfield zones — leading to multi-crore over-investment in roads or BRT corridors that don't materialise.


## Step 1 — Problem Type

| Decision | Choice | Why |
|---|---|---|
| Labels available? | No | Even if planners had a tagging, we want data-driven clusters |
| Family | Unsupervised | We're discovering structure, not predicting |
| Algorithms compared | K-Means, Hierarchical (Ward), DBSCAN, Gaussian Mixture | Spans centroid, hierarchical, density, and probabilistic methods |
| Dimensionality | 6 features | Manageable, but PCA still useful for visualisation |


## Step 2 — Data Collection

**Real sources for socio-economic zonal data:**

| Variable | Source |
|---|---|
| Population | Census of India 2011 (next: 2027) [5] |
| Households | Census of India 2011 [5] |
| Employment | NSSO PLFS / Economic Census [6] |
| Retail employment | Economic Census 2013 (next due) [6] |
| Students enrolled | UDISE+ school-level data [7] |
| Median income | NSS Consumption Expenditure / private estimates |
| Zone boundaries | OpenStreetMap + city CMP maps |

For this notebook we use a synthetic 15-TAZ city with realistic ranges drawn from these sources. To swap in a real city, the only change is the input CSV.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

taz = pd.read_csv('../data/case3_taz.csv')
print(f'TAZ shape: {taz.shape}')
print(taz.describe().round(0))
taz.head()

## Step 3 — Exploratory Data Analysis

### 3.1 Distributions and skewness

Socio-economic variables are typically right-skewed (a few high-employment CBD zones, many lower-employment residential zones). We log-transform skewed variables before clustering — distance-based algorithms like K-Means are dominated by high-magnitude features otherwise.


In [ ]:
features = ['population', 'households', 'employment', 'retail_employment',
            'students_enrolled', 'median_hh_income']

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.flat, features):
    taz[col].hist(ax=ax, bins=10, color='#1F3864', edgecolor='white')
    ax.set_title(f'{col}\nskew = {taz[col].skew():.2f}')
plt.tight_layout()
plt.show()

### 3.2 Correlation analysis

Same rule as Notebooks 1 and 2 — pairs with `|r| > 0.85` lose one member. For clustering, redundant features get **double weight** in the distance metric, which is exactly what we don't want.


In [ ]:
corr = taz[features].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature correlations')
plt.tight_layout()
plt.show()

# Identify pairs to drop
print('\n|r| > 0.85 pairs:')
for i in range(len(features)):
    for j in range(i+1, len(features)):
        r = corr.iloc[i, j]
        if abs(r) > 0.85:
            print(f'  {features[i]:20s} <-> {features[j]:20s} r = {r:+.3f}')

**Decision:** `population` and `households` are mechanically related (households = pop / family_size). We keep `households` (more directly linked to trip productions in the trip-generation literature [1]) and drop `population`.

`employment` and `retail_employment` are positively correlated but capture different attractions (work trips vs shopping trips). We keep both.

### 3.3 Pairplot for visual class structure

Even before clustering, a pairplot can reveal natural groups.


In [ ]:
features_used = ['households', 'employment', 'retail_employment',
                 'students_enrolled', 'median_hh_income']

# Quick pairplot — small enough dataset to be useful
sns.pairplot(taz[features_used], height=1.5, plot_kws={'alpha': 0.7, 's': 30})
plt.suptitle('Pairwise feature relationships', y=1.02)
plt.show()

## Step 4 & 5 — Cleaning, Preprocessing, Feature Engineering

### Why standardisation is non-negotiable for clustering

K-Means minimises the Within-Cluster Sum of Squares (WCSS):

```
WCSS = Σ_clusters Σ_{x in cluster} ||x − centroid||²
```

The Euclidean distance `||x − c||` is dominated by features with large numerical ranges. `median_hh_income` (range: 25,000–120,000) would crush `retail_employment` (range: 500–20,000) without scaling. We use **standardisation**:

```
x_scaled = (x − mean) / std
```

After this, every feature has mean 0 and std 1, contributing equally to the distance.


In [ ]:
from sklearn.preprocessing import StandardScaler

X = taz[features_used].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Before scaling — means:', X.mean().round(0).to_dict())
print('After scaling — means:', np.round(X_scaled.mean(axis=0), 2))
print('After scaling — stds:',  np.round(X_scaled.std(axis=0), 2))

## Step 6 & 7 — Try Four Clustering Algorithms

### A. K-Means

**Algorithm (Lloyd's method):**

```
1. Choose k (number of clusters)
2. Initialise k random centroids (k-means++ does smart initialisation)
3. Repeat until convergence:
   a. Assign each point to its nearest centroid
   b. Recompute each centroid as the mean of its assigned points
```

**Choosing k — the Elbow Method:**

```
Plot WCSS vs k. Pick k at the "elbow" where adding another cluster
doesn't reduce WCSS much.
```

**Choosing k — Silhouette Coefficient:**

```
For each point i:
  a_i = mean distance to other points in its own cluster
  b_i = mean distance to points in the nearest other cluster
  s_i = (b_i − a_i) / max(a_i, b_i)        ∈ [−1, +1]

Silhouette = mean(s_i) over all points.   Higher = better.
```

A silhouette > 0.5 indicates strong cluster structure; 0.25–0.5 reasonable; below 0.25 weak [8].


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Elbow method + silhouette
ks = range(2, 9)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(ks), wcss, 'o-', color='#1F3864')
axes[0].set_xlabel('k'); axes[0].set_ylabel('WCSS (inertia)')
axes[0].set_title('Elbow method')
axes[0].grid(alpha=0.3)

axes[1].plot(list(ks), sil, 'o-', color='#C00000')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette score')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Pick the k with best silhouette
best_k = ks[int(np.argmax(sil))]
print(f'Best k (highest silhouette): {best_k}  (silhouette = {max(sil):.3f})')

### B. Hierarchical Clustering (Agglomerative, Ward linkage)

**Algorithm:**

```
1. Start with each point as its own cluster (n clusters total)
2. At each step, merge the two clusters that produce the smallest
   increase in total within-cluster variance (Ward linkage)
3. Continue until only 1 cluster remains, recording the merge tree
4. Cut the tree at the desired k
```

The output is a **dendrogram** — a tree showing the merge order. Where you "cut" the dendrogram determines the number of clusters.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, labels=taz['taz_id'].astype(str).values, ax=ax,
           color_threshold=0.7 * max(Z[:, 2]))
ax.set_title('Hierarchical clustering — Ward dendrogram')
ax.set_xlabel('TAZ ID'); ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

# Cut at the same k
hc_labels = fcluster(Z, t=best_k, criterion='maxclust') - 1  # 0-indexed
print(f'Hierarchical at k={best_k} silhouette: {silhouette_score(X_scaled, hc_labels):.3f}')

### C. DBSCAN — Density-Based

DBSCAN doesn't require choosing k. It finds clusters by density: a point is part of a cluster if it has at least `min_samples` neighbours within distance `eps`. Outliers are labelled `-1`.

**Algorithm:**

```
For each unvisited point p:
  Find p's neighbours within eps
  If len(neighbours) >= min_samples:
    Start a new cluster, expand it through density-reachable points
  Else:
    Label p as noise (may be re-assigned later if reachable from a core point)
```

DBSCAN excels when clusters have irregular shapes or when you actually want to detect outliers (think: zones with anomalous data that should be flagged for re-survey).


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# Choose eps using k-nearest-neighbour distance plot
k_nn = 4
nbrs = NearestNeighbors(n_neighbors=k_nn).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances_sorted = np.sort(distances[:, k_nn - 1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(distances_sorted)
ax.set_xlabel('Points (sorted)'); ax.set_ylabel(f'Distance to {k_nn}-th NN')
ax.set_title('k-distance plot — pick eps at the "knee"')
ax.grid(alpha=0.3)
plt.show()

# Try eps = 75th percentile of the k-distances
eps_choice = np.percentile(distances_sorted, 75)
db = DBSCAN(eps=eps_choice, min_samples=k_nn).fit(X_scaled)
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise = (db.labels_ == -1).sum()
print(f'DBSCAN with eps={eps_choice:.2f}: {n_clusters} clusters, {n_noise} noise points')

### D. Gaussian Mixture Model (GMM)

GMM models the data as a mixture of `k` Gaussian distributions. Unlike K-Means' hard assignments, GMM gives **soft** probabilities — useful when zone identity isn't strictly binary.

**Algorithm (Expectation-Maximisation):**

```
E-step: For each point, compute the probability it belongs to each Gaussian
M-step: Update each Gaussian's mean and covariance to maximise the likelihood
        of the assigned probabilities
Repeat until log-likelihood converges
```

**Choosing k — BIC (Bayesian Information Criterion):**

```
BIC = -2·log(L) + p·log(n)
```

where `L` is the likelihood, `p` the number of parameters, `n` the sample size. Lower BIC is better — it penalises both poor fit and excessive complexity.


In [ ]:
from sklearn.mixture import GaussianMixture

ks = range(2, 8)
bics, aics, sils = [], [], []
for k in ks:
    gm = GaussianMixture(n_components=k, random_state=42, covariance_type='full')
    gm.fit(X_scaled)
    bics.append(gm.bic(X_scaled))
    aics.append(gm.aic(X_scaled))
    sils.append(silhouette_score(X_scaled, gm.predict(X_scaled)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(ks), bics, 'o-', label='BIC', color='#1F3864')
ax.plot(list(ks), aics, 's-', label='AIC', color='#C00000')
ax.set_xlabel('k'); ax.set_ylabel('Information criterion')
ax.set_title('Model selection for GMM')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

best_k_gmm = ks[int(np.argmin(bics))]
print(f'Best k by BIC: {best_k_gmm}')
gmm = GaussianMixture(n_components=best_k_gmm, random_state=42).fit(X_scaled)
gmm_labels = gmm.predict(X_scaled)

## Step 8 — Evaluation: Compare All Four Algorithms

### Metrics for unsupervised problems

Without ground-truth labels, we use intrinsic metrics:

```
Silhouette         (higher is better, range [-1, 1])
Davies-Bouldin     (lower is better — ratio of within-cluster vs between-cluster scatter)
Calinski-Harabasz  (higher is better — variance ratio)
```

A model that scores well on multiple metrics is more trustworthy than one that wins on a single metric.


In [ ]:
from sklearn.metrics import calinski_harabasz_score

def evaluate_clustering(name, labels, X):
    # Filter out noise points (label -1) for silhouette
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        return {'Algorithm': name, 'Silhouette': np.nan,
                'Davies-Bouldin': np.nan, 'Calinski-Harabasz': np.nan,
                'n_clusters': len(set(labels[mask])), 'n_noise': (~mask).sum()}
    return {
        'Algorithm': name,
        'Silhouette': silhouette_score(X[mask], labels[mask]),
        'Davies-Bouldin': davies_bouldin_score(X[mask], labels[mask]),
        'Calinski-Harabasz': calinski_harabasz_score(X[mask], labels[mask]),
        'n_clusters': len(set(labels[mask])),
        'n_noise': (~mask).sum(),
    }

# Final fits
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_scaled)
hc_labels_final = fcluster(linkage(X_scaled, 'ward'), t=best_k, criterion='maxclust') - 1

results = pd.DataFrame([
    evaluate_clustering('K-Means',       km_final.labels_,  X_scaled),
    evaluate_clustering('Hierarchical',  hc_labels_final,    X_scaled),
    evaluate_clustering('DBSCAN',        db.labels_,         X_scaled),
    evaluate_clustering('GMM',           gmm_labels,         X_scaled),
]).set_index('Algorithm').round(3)
print('Clustering comparison:')
print(results)

## Step 9 — Visualise in 2D using PCA

PCA finds the directions of maximum variance. Projecting our 5-dimensional standardised data onto the top-2 principal components lets us *see* whether the clusters look reasonable.

**Math:**

```
1. Compute the covariance matrix Σ = X^T X / (n-1)
2. Find eigenvalues λ_1 >= λ_2 >= ... and eigenvectors v_1, v_2, ...
3. Project: PC_i = X · v_i
4. % variance explained by PC_i = λ_i / Σ λ_j
```


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
print(f'PC1 variance: {explained[0]:.1%}, PC2: {explained[1]:.1%}, total: {explained.sum():.1%}')

# 2x2 grid showing all 4 algorithms
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
labelings = {
    'K-Means':       km_final.labels_,
    'Hierarchical':  hc_labels_final,
    'DBSCAN':        db.labels_,
    'GMM':           gmm_labels,
}
for ax, (name, labels) in zip(axes.flat, labelings.items()):
    sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10',
                    s=120, edgecolor='k', linewidth=0.5)
    for i, txt in enumerate(taz['taz_id']):
        ax.annotate(txt, (X_pca[i, 0], X_pca[i, 1]), fontsize=7,
                    xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel(f'PC1 ({explained[0]:.0%})')
    ax.set_ylabel(f'PC2 ({explained[1]:.0%})')
    ax.set_title(f'{name} (k = {len(set(labels)) - (1 if -1 in labels else 0)})')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10 & 11 — Profile and Name the Clusters

The metric tells us *how distinct* the clusters are; the **profile** tells us *what they mean*.

For our chosen K-Means solution, we compute the mean of each feature within each cluster, then compare to the city-wide mean. Features where a cluster sits well above or below the city mean give it its identity.


In [ ]:
# Use the best K-Means solution
taz['cluster'] = km_final.labels_
profile = taz.groupby('cluster')[features_used].mean().round(0)
city_mean = taz[features_used].mean()

# Z-score profile: how many city-stds above/below the mean is each cluster?
profile_z = (profile - city_mean) / taz[features_used].std()
print('Cluster centroids (raw values):')
print(profile)
print('\nCluster z-scores (how many stds above/below city mean):')
print(profile_z.round(2))
print('\nCluster sizes:')
print(taz['cluster'].value_counts().sort_index())

# Visual summary
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(profile_z, annot=True, fmt='+.2f', cmap='RdBu_r', center=0, ax=ax,
            cbar_kws={'label': 'Standard deviations from city mean'})
ax.set_title('Cluster profiles (z-scored)')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

### Naming the clusters — interpretation step

This is where domain knowledge enters. Looking at the z-score heatmap above, the planner reads each row:

- A cluster with **high employment + high retail employment + lower households** → "Commercial / CBD"
- A cluster with **high households + high students + moderate employment** → "Mixed-use residential"
- A cluster with **high households + low employment + low retail** → "Pure residential"
- A cluster with **moderate everything** → "Suburban / mixed"
- A cluster with **high income + low employment** → "Affluent residential"

The exact names depend on what the model produced. The point is: clusters carry meaning only when a planner names them.

## Step 12 — Deployment

This isn't a real-time predictive model — it's a **planning artefact**. The cluster assignment becomes part of the TAZ shapefile delivered to the CMP team:

```
TAZ_01 ... cluster_label = "CBD"        → use coefficient set α^CBD
TAZ_02 ... cluster_label = "Residential" → use coefficient set α^Res
```

These coefficient sets are calibrated separately on each cluster's data, and feed into Stage 1 of the 4-stage model.

## Step 13 — Maintenance

| Trigger | Action |
|---|---|
| New Census data | Re-cluster — boundaries shift over a decade |
| New large land-use change (e.g., new airport, new IT park) | Manually reassign affected zones, then re-cluster |
| Sub-cluster appears (e.g., "transit-oriented residential") | Increase k, re-evaluate |

---

## Business Takeaway

| Question | Answer |
|---|---|
| Best algorithm | K-Means usually wins on this kind of clean tabular socio-economic data; GMM adds soft probabilities if you need them |
| Cluster count | Driven by silhouette + planner judgement, not by mathematics alone |
| Where this fits | Stage 1 (Trip Generation) of the classical 4-stage transport model — directly used in CMPs by WRI [3] and ITDP [4] |
| Beyond planning | Same technique segments customers in retail (Uber rider segments), buildings in real estate, parcels in logistics |


## References

[1] Ortúzar, J. de D. & Willumsen, L. G. *Modelling Transport.* 4th ed., Wiley, 2011.

[2] Cascetta, E. *Transportation Systems Analysis: Models and Applications.* 2nd ed., Springer, 2009.

[3] WRI India. *Comprehensive Mobility Plans — Practitioner's Guide.* <https://wri-india.org/sustainable-cities>

[4] Institute for Transportation and Development Policy (ITDP). *Better Streets, Better Cities — Planning Guides.* <https://www.itdp.org/library/standards-and-guides/>

[5] Office of the Registrar General & Census Commissioner, India. *Census of India.* <https://censusindia.gov.in/>

[6] Ministry of Statistics and Programme Implementation (MoSPI). *Periodic Labour Force Survey (PLFS).* <https://www.mospi.gov.in/>

[7] Department of School Education & Literacy. *Unified District Information System for Education Plus (UDISE+).* <https://udiseplus.gov.in/>

[8] Rousseeuw, P. J. *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis.* Journal of Computational and Applied Mathematics, 1987.

[9] Ester, M., Kriegel, H.-P., Sander, J. & Xu, X. *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise (DBSCAN).* KDD 1996.

[10] Pedregosa, F. et al. *Scikit-learn: Machine Learning in Python.* JMLR, 2011. <https://scikit-learn.org/>
